# PROJECT FORESIGHT: 02 Baseline Demand Forecasting
**Methodology:** 4-Week Moving Average with Temporal Validation


## 1. Setup & Imports


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast import aggregate_to_weekly, baseline_forecast, evaluate_model


## 2. Load & Aggregate Weekly Demand


In [ ]:
df_merged = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'merged_analysis_ready.csv')
weekly_df = aggregate_to_weekly(df_merged)
print(f'Weekly aggregated sales shape: {weekly_df.shape}')
weekly_df.head()


## 3. Temporal Train/Test Split (Last 8 Weeks Horizon)


In [ ]:
max_date = weekly_df['week_start_date'].max()
cutoff_date = max_date - pd.Timedelta(weeks=8)

train_df = weekly_df[weekly_df['week_start_date'] <= cutoff_date]
test_df = weekly_df[weekly_df['week_start_date'] > cutoff_date]

print(f'Train period: {train_df["week_start_date"].min()} to {train_df["week_start_date"].max()} ({len(train_df)} rows)')
print(f'Test period:  {test_df["week_start_date"].min()} to {test_df["week_start_date"].max()} ({len(test_df)} rows)')


## 4. Run 4-Week Moving Average Baseline


In [ ]:
base_preds = baseline_forecast(train_df, horizon=8)
test_merged = test_df.merge(base_preds.rename(columns={'week_start': 'week_start_date'}), on=['sku_id', 'week_start_date'], how='inner')

metrics = evaluate_model(test_merged['weekly_units'].values, test_merged['forecast_units'].values)
print('Baseline Performance on Test Horizon:')
for k, v in metrics.items():
    print(f'  {k.upper()}: {v:.2f}')


## 5. Sample SKU Baseline Forecast Visualization


In [ ]:
sample_sku = 'SKU-001'
sku_actual = weekly_df[weekly_df['sku_id'] == sample_sku]
sku_base = base_preds[base_preds['sku_id'] == sample_sku]

plt.figure(figsize=(12, 5))
plt.plot(pd.to_datetime(sku_actual['week_start_date']), sku_actual['weekly_units'], label='Actual Sales', marker='o')
plt.plot(pd.to_datetime(sku_base['week_start']), sku_base['forecast_units'], label='4-Week MA Baseline', linestyle='--', color='orange', marker='s')
plt.title(f'Baseline Forecast vs Actuals for {sample_sku}')
plt.xlabel('Week Start')
plt.ylabel('Weekly Units')
plt.legend()
plt.show()
